# Clean Train and Test Datasets Creation

In [ ]:
import os
import shutil
import random
from pathlib import Path
import torch
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

from data_loader.dataset import EuroSatDataset

In [ ]:
def create_and_save_datasets(
    data_path,
    save_path,
    train_ratio=0.8,
    seed=42
):
    random.seed(seed)

    data_path = Path(data_path)
    save_path = Path(save_path)

    train_dir = save_path / "train_clean"
    test_dir = save_path / "test_clean"

    # Remove existing directories if they exist
    if train_dir.exists():
        print(f"Removing existing directory: {train_dir}")
        shutil.rmtree(train_dir)
    
    if test_dir.exists():
        print(f"Removing existing directory: {test_dir}")
        shutil.rmtree(test_dir)

    classes = sorted([d for d in data_path.iterdir() if d.is_dir()])

    print(f"Found {len(classes)} classes")

    for class_dir in classes:
        class_name = class_dir.name

        # Créer les dossiers de sortie
        (train_dir / class_name).mkdir(exist_ok=True)
        (test_dir / class_name).mkdir(exist_ok=True)

        # Lister les images
        images = sorted(
            [img for img in list(class_dir.glob("*"))
            if img.suffix.lower() in [".jpg", ".png", ".jpeg"]]
        )

        random.Random(seed).sample(images, len(images))

        n_train = int(len(images) * train_ratio)

        train_images = images[:n_train]
        test_images = images[n_train:]

        # Copier les fichiers
        for img_path in train_images:
            shutil.copy(img_path, train_dir / class_name / img_path.name)

        for img_path in test_images:
            shutil.copy(img_path, test_dir / class_name / img_path.name)

        print(
            f"Class {class_name}: "
            f"{len(train_images)} train / {len(test_images)} test"
        )

    print("\nDataset split completed.")
    print(f"Train directory: {train_dir}")
    print(f"Test directory: {test_dir}")


create_and_save_datasets(data_path="data/EuroSAT_RGB", save_path="datasets/EuroSAT_RGB", train_ratio=0.8, seed=42)

In [ ]:
def compute_stats():
    train_path = "./datasets/EuroSAT_RGB/train_clean"
    
    transform = transforms.Compose([
        transforms.Resize((64, 64)),
        transforms.ToTensor()
    ])
    
    # Loads only train set
    dataset = EuroSatDataset(
        root_dir=train_path,
        transform=transform,
        train=True
    )
    
    dataloader = DataLoader(dataset, batch_size=64, shuffle=False, num_workers=4)
    
    mean = 0.
    std = 0.
    nb_samples = 0.
    
    for data, _ in dataloader:
        # data: [batch, 3, H, W]
        batch_samples = data.size(0)
        data = data.view(batch_samples, data.size(1), -1)
        
        mean += data.mean(2).sum(0)
        std += data.std(2).sum(0)
        nb_samples += batch_samples
    
    mean /= nb_samples
    std /= nb_samples
    
    print(f"MEAN = {mean.tolist()}")
    print(f"STD = {std.tolist()}")
    
    return mean.tolist(), std.tolist()


def save_statistics_to_config(mean, std, config_file_path="config.py"):
    """
    Saves statistics into config file.
    
    Args:
        mean: Mean list [R, G, B]
        std: Standard errors list [R, G, B]
        config_file_path: Path to config.py
    """
    
    with open(config_file_path, 'r') as f:
        lines = f.readlines()
    
    new_lines = []
    for line in lines:
        if line.strip().startswith("MEAN ="):
            new_lines.append(f"MEAN = {mean}  # Computed on train set\n")
        elif line.strip().startswith("STD ="):
            new_lines.append(f"STD = {std}  # Computed on train set\n")
        else:
            new_lines.append(line)
    
    # Écrire le fichier mis à jour
    with open(config_file_path, 'w') as f:
        f.writelines(new_lines)
    
    print(f"Statistiques sauvegardées dans {config_file_path}")


mean, std = compute_stats()
save_statistics_to_config(mean, std, config_file_path="config.py")

# Baseline Model Creation

In [ ]:
import sys

sys.argv = [
    "main.py",
    "--model", "resnet18",
    "--train",
    "--evaluate",
    "--visualize",
    "--epochs", "50",
    "--patience", "20",
    "--lr", "0.001",
    "--batch-size", "32",
    "--seed", "42",
    "--data-path-train", "datasets/EuroSAT_RGB/train_clean",
    "--data-path-eval", "datasets/EuroSAT_RGB/test_clean",
    "--save-model-path", "outputs/models/baseline",
    "--save-plots-path", "outputs/plots/baseline_clean",
]

from main import main
main()

# Adversarial Test Dataset Creation

In [ ]:
import os
import torch
from torch.utils.data import DataLoader
from torchvision.utils import save_image
from tqdm import tqdm
import json

from models import ResNet18
from data_loader.dataset import EuroSatDataset
from attacks.pgd import PGD
from config import BATCH_SIZE, DEVICE, MEAN, STD, SEED

In [ ]:
def load_model_and_create_attacked_test_dataset(
    model_path='outputs/models/best_model.pth',
    test_clean_path='datasets/EuroSAT_RGB/test_clean',
    save_adv_path='datasets/EuroSAT_RGB/test_pgd_eps002',
    epsilon_pixel=0.02,
    alpha_pixel=0.004,
    iterations=5
):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    # Load model
    model = ResNet18().to(device)

    if os.path.exists(model_path):
        checkpoint = torch.load(model_path, map_location=device)
        if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
            model.load_state_dict(checkpoint['model_state_dict'])
        else:
            model.load_state_dict(checkpoint)
        print(f"Model loaded from {model_path}")
    else:
        raise FileNotFoundError(f"Model not found at {model_path}")

    model.eval()

    # Load clean test dataset
    dataset = EuroSatDataset(
        root_dir=test_clean_path,
        train=False
    )

    class_names = dataset.classes

    test_loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=2
    )

    # Prepare save folders
    os.makedirs(save_adv_path, exist_ok=True)
    for cls in class_names:
        os.makedirs(os.path.join(save_adv_path, cls), exist_ok=True)

    # PGD attack
    epsilon = torch.tensor([epsilon_pixel for _ in range(3)]).view(1,3,1,1)
    alpha   = torch.tensor([alpha_pixel for _ in range(3)]).view(1,3,1,1)

    pgd = PGD(
        model=model,
        epsilon=epsilon,
        alpha=alpha,
        iterations=iterations,
        random_start=True,
        device=device,
        seed=SEED,
    )

    # Generate & save adversarial images
    img_idx = 0

    for images, labels in tqdm(test_loader, desc="Generating adversarial test set"):
        images = images.to(device)
        labels = labels.to(device)

        with torch.enable_grad():
            adv_images = pgd.attack(images, labels)

        for i in range(adv_images.size(0)):
            label = labels[i].item()
            class_name = class_names[label]

            save_path = os.path.join(
                save_adv_path,
                class_name,
                f"img_{img_idx}.png"
            )

            save_image(adv_images[i], save_path)
            img_idx += 1

    print(f"\nAdversarial test dataset saved to: {save_adv_path}")

    
    attack_config = {
        "attack": "PGD",
        "epsilon": epsilon.tolist() if torch.is_tensor(epsilon) else epsilon,
        "alpha": alpha.tolist() if torch.is_tensor(alpha) else alpha,
        "iterations": iterations,
        "random_start": True,
        "seed": SEED,
        "normalization": {
            "mean": MEAN,
            "std": STD
        }
    }

    os.makedirs("attacks/configs", exist_ok=True)
    with open(os.path.join("attacks/configs", "attack_config.json"), "w") as f:
        json.dump(attack_config, f, indent=4)

    return None

load_model_and_create_attacked_test_dataset(
    model_path='outputs/models/baseline/best_model.pth',
    test_clean_path='datasets/EuroSAT_RGB/test_clean',
    save_adv_path='datasets/EuroSAT_RGB/test_pgd_eps002',
    epsilon_pixel=0.02,
    alpha_pixel=0.004,
    iterations=5
)

In [ ]:
import sys

sys.argv = [
    "main.py",
    "--model", "resnet18",
    "--evaluate",
    "--visualize",
    "--seed", "42",
    "--data-path-eval", "datasets/EuroSAT_RGB/test_pgd_eps002",
    "--save-plots-path", "outputs/plots/baseline_pgd_eps002",
    "--save-model-path", "outputs/model/baseline",
]

from main import main
main()